In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [23]:
df = pd.read_csv('../data/data_with_polysemy.csv')
df = df.rename(columns={
    "DATA_FILE": "subject_id",
    "IA_DWELL_TIME": "reading_time",
    "V.Mean.Sum": "valence",
    "A.Mean.Sum": "arousal",
    "D.Mean.Sum": "dominance"
})

In [24]:
df = df.replace([np.inf, -np.inf], np.nan).dropna()
df["participant_sentence"] = df["subject_id"].astype(str) + "_" + df["sentence"].astype(str)

Q1 = df["reading_time"].quantile(0.25)
Q3 = df["reading_time"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
lower_bound = lower_bound if lower_bound > 0 else 0
upper_bound = Q3 + 1.5 * IQR
print(f"lower_bound: {lower_bound}")
print(f"upper_bound: {upper_bound}")

lower_bound: 0
upper_bound: 1514.5


In [25]:
all_features = ["SURP_GPT2", "WORD_LEN", "FREQ_WEB", "valence", "arousal", "dominance", "orth_size", "polysemy_count", "answered_correctly"]
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["reading_time"] + all_features)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Linear Regression

## BaseLine (Surprisal, Length)

In [26]:
model_lm_bl = smf.ols("reading_time ~ SURP_GPT2 + WORD_LEN", data=train_df).fit()
print(model_lm_bl.summary())

                            OLS Regression Results                            
Dep. Variable:           reading_time   R-squared:                       0.165
Model:                            OLS   Adj. R-squared:                  0.165
Method:                 Least Squares   F-statistic:                 4.756e+04
Date:                Thu, 31 Jul 2025   Prob (F-statistic):               0.00
Time:                        14:07:56   Log-Likelihood:            -3.7310e+06
No. Observations:              481640   AIC:                         7.462e+06
Df Residuals:                  481637   BIC:                         7.462e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     61.1224      1.770     34.532      0.0

## All Features

In [27]:
model_lm_all = smf.ols("reading_time ~ SURP_GPT2 + WORD_LEN + FREQ_WEB + valence + arousal + dominance + orth_size + polysemy_count + answered_correctly", data=train_df).fit()
print(model_lm_all.summary())

                            OLS Regression Results                            
Dep. Variable:           reading_time   R-squared:                       0.174
Model:                            OLS   Adj. R-squared:                  0.174
Method:                 Least Squares   F-statistic:                 1.126e+04
Date:                Thu, 31 Jul 2025   Prob (F-statistic):               0.00
Time:                        14:08:11   Log-Likelihood:            -3.7284e+06
No. Observations:              481640   AIC:                         7.457e+06
Df Residuals:                  481630   BIC:                         7.457e+06
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              6.8829     10

## Model Selection (Backward StepWise Regression)

In [34]:
# Initial full set of features
features = [
    "FREQ_WEB", "WORD_LEN", "SURP_GPT2", "valence", "arousal",
    "dominance", "orth_size", "polysemy_count", "answered_correctly"
]

# Set initial best model
best_features = features.copy()
current_best_aic = model_lm_all.aic
improvement = True

while improvement and len(best_features) > 1:
    aic_scores = []
    models = []

    for feature_to_remove in best_features:
        trial_features = [f for f in best_features if f != feature_to_remove]
        formula = "reading_time ~ " + " + ".join(trial_features)

        try:
            md = smf.ols(formula, train_df)
            model = md.fit()
            aic_scores.append((model.aic, feature_to_remove, model))
        except Exception as e:
            # If model fails to converge or fit, skip
            print(f"Model failed without {feature_to_remove}: {e}")

    # Find the model with the lowest AIC
    aic_scores.sort()
    best_aic, feature_removed, best_model = aic_scores[0]

    if best_aic < current_best_aic:
        print(f"Removing {feature_removed} improved AIC: {current_best_aic:.2f} → {best_aic:.2f}")
        current_best_aic = best_aic
        best_features.remove(feature_removed)
        model_lm_sm = best_model  # Update model
    else:
        print("No further improvement in AIC.")
        improvement = False

# Final selected model summary
print("\nFinal selected features:", best_features)
print(model_lm_sm.summary())

Removing polysemy_count improved AIC: 7456829.19 → 7456827.26
No further improvement in AIC.

Final selected features: ['FREQ_WEB', 'WORD_LEN', 'SURP_GPT2', 'valence', 'arousal', 'dominance', 'orth_size', 'answered_correctly']
                            OLS Regression Results                            
Dep. Variable:           reading_time   R-squared:                       0.174
Model:                            OLS   Adj. R-squared:                  0.174
Method:                 Least Squares   F-statistic:                 1.266e+04
Date:                Thu, 31 Jul 2025   Prob (F-statistic):               0.00
Time:                        14:14:48   Log-Likelihood:            -3.7284e+06
No. Observations:              481640   AIC:                         7.457e+06
Df Residuals:                  481631   BIC:                         7.457e+06
Df Model:                           8                                         
Covariance Type:            nonrobust                         

## Comparison

In [35]:
comparison_df = pd.DataFrame({
    "Model": ["Only surprisal+length", "All features", "Model Selection"],
    "AIC": [model_lm_bl.aic, model_lm_all.aic, model_lm_sm.aic],
    "BIC": [model_lm_bl.bic, model_lm_all.bic, model_lm_sm.bic],
    "Log-Likelihood": [model_lm_bl.llf, model_lm_all.llf, model_lm_sm.llf]
})

print(comparison_df)

                   Model           AIC           BIC  Log-Likelihood
0  Only surprisal+length  7.461948e+06  7.461981e+06   -3.730971e+06
1           All features  7.456829e+06  7.456940e+06   -3.728405e+06
2        Model Selection  7.456827e+06  7.456927e+06   -3.728405e+06


In [36]:
def evaluate_model(model, test_df):
    y_true = test_df["reading_time"]
    y_pred = model.predict(test_df)

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {"MSE": mse, "MAE": mae, "R2": r2}

results = {
    "BaseLine": evaluate_model(model_lm_bl, test_df),
    "All Features": evaluate_model(model_lm_all, test_df),
    "Model Selection": evaluate_model(model_lm_sm, test_df)
}

results_df = pd.DataFrame(results).T
print(results_df)

                           MSE         MAE        R2
BaseLine         321132.541176  364.599486  0.164515
All Features     317291.660147  362.172590  0.174508
Model Selection  317291.591898  362.174795  0.174508


# Mixes Effect Model

## BaseLine (Surprisal, Length)

In [40]:
model_mixed_bl = smf.mixedlm(
    "reading_time ~ SURP_GPT2 + WORD_LEN",
    train_df,
    groups=train_df["subject_id"],  # or the grouping variable you're using
    vc_formula={"sentence": "0 + C(sentence)"}  # if you want sentence as random intercept
).fit(reml=False)
print(model_mixed_bl.summary())

           Mixed Linear Model Regression Results
Model:             MixedLM Dependent Variable: reading_time 
No. Observations:  481640  Method:             ML           
No. Groups:        365     Scale:              192007.8115  
Min. group size:   1238    Log-Likelihood:     -3663968.3951
Max. group size:   1410    Converged:          Yes          
Mean group size:   1319.6                                   
------------------------------------------------------------
               Coef.    Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept        57.133    2.081  27.455 0.000 53.054 61.211
SURP_GPT2        17.982    0.149 120.706 0.000 17.690 18.274
WORD_LEN         74.723    0.307 243.612 0.000 74.122 75.324
sentence Var 121414.403    2.121                            



## All Features

In [41]:
model_mixed_all = smf.mixedlm(
    "reading_time ~ SURP_GPT2 + WORD_LEN + FREQ_WEB + valence + arousal + dominance + orth_size + polysemy_count + answered_correctly",
    train_df,
    groups=train_df["subject_id"],  # or the grouping variable you're using
    vc_formula={"sentence": "0 + C(sentence)"}  # if you want sentence as random intercept
).fit(reml=False)
print(model_mixed_all.summary())

               Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    reading_time 
No. Observations:    481640     Method:                ML           
No. Groups:          365        Scale:                 189037.3887  
Min. group size:     1238       Log-Likelihood:        -3660499.4962
Max. group size:     1410       Converged:             Yes          
Mean group size:     1319.6                                         
--------------------------------------------------------------------
                     Coef.    Std.Err.    z    P>|z|  [0.025  0.975]
--------------------------------------------------------------------
Intercept              31.681   10.165   3.117 0.002  11.758  51.604
SURP_GPT2              11.524    0.171  67.373 0.000  11.189  11.859
WORD_LEN              106.964    1.853  57.737 0.000 103.332 110.595
FREQ_WEB               21.403    0.287  74.575 0.000  20.840  21.965
valence               -13.026    1.205 -10.807 0.0

## Model Selection (Backward StepWise Regression)

In [44]:
# Initial full set of features
features = [
    "FREQ_WEB", "WORD_LEN", "SURP_GPT2", "valence", "arousal",
    "dominance", "orth_size", "polysemy_count", "answered_correctly"
]

# Set initial best model
best_features = features.copy()
current_best_aic = model_mixed_all.aic
improvement = True

while improvement and len(best_features) > 1:
    aic_scores = []
    models = []

    for feature_to_remove in best_features:
        trial_features = [f for f in best_features if f != feature_to_remove]
        formula = "reading_time ~ " + " + ".join(trial_features)

        try:
            md = smf.mixedlm(formula, train_df, groups="subject_id",
                             vc_formula={"sentence": "0 + C(sentence)"})
            model = md.fit(reml=False)
            aic_scores.append((model.aic, feature_to_remove, model))
        except Exception as e:
            # If model fails to converge or fit, skip
            print(f"Model failed without {feature_to_remove}: {e}")

    # Find the model with the lowest AIC
    aic_scores.sort()
    best_aic, feature_removed, best_model = aic_scores[0]

    if best_aic < current_best_aic:
        print(f"Removing {feature_removed} improved AIC: {current_best_aic:.2f} → {best_aic:.2f}")
        current_best_aic = best_aic
        best_features.remove(feature_removed)
        model_mixed_sm = best_model  # Update model
    else:
        print("No further improvement in AIC.")
        improvement = False

# Final selected model summary
print("\nFinal selected features:", best_features)
print(model_mixed_sm.summary())

Removing dominance improved AIC: 7321022.99 → 7321021.06
No further improvement in AIC.

Final selected features: ['FREQ_WEB', 'WORD_LEN', 'SURP_GPT2', 'valence', 'arousal', 'orth_size', 'polysemy_count', 'answered_correctly']
               Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    reading_time 
No. Observations:    481640     Method:                ML           
No. Groups:          365        Scale:                 189037.6651  
Min. group size:     1238       Log-Likelihood:        -3660499.5311
Max. group size:     1410       Converged:             Yes          
Mean group size:     1319.6                                         
--------------------------------------------------------------------
                     Coef.    Std.Err.    z    P>|z|  [0.025  0.975]
--------------------------------------------------------------------
Intercept              30.760    9.548   3.222 0.001  12.047  49.474
FREQ_WEB               21.397 

## Comparison

In [45]:
comparison_df = pd.DataFrame({
    "Model": ["Only surprisal+length", "All features", "Model Selection"],
    "AIC": [model_mixed_bl.aic, model_mixed_all.aic, model_mixed_sm.aic],
    "BIC": [model_mixed_bl.bic, model_mixed_all.bic, model_mixed_sm.bic],
    "Log-Likelihood": [model_mixed_bl.llf, model_mixed_all.llf, model_mixed_sm.llf]
})

print(comparison_df)

                   Model           AIC           BIC  Log-Likelihood
0  Only surprisal+length  7.327947e+06  7.328002e+06   -3.663968e+06
1           All features  7.321023e+06  7.321156e+06   -3.660499e+06
2        Model Selection  7.321021e+06  7.321143e+06   -3.660500e+06


In [46]:
def evaluate_model(model, test_df):
    y_true = test_df["reading_time"]
    y_pred = model.predict(test_df)

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {"MSE": mse, "MAE": mae, "R2": r2}

results = {
    "BaseLine": evaluate_model(model_mixed_bl, test_df),
    "All Features": evaluate_model(model_mixed_all, test_df),
    "Model Selection": evaluate_model(model_mixed_sm, test_df)
}

results_df = pd.DataFrame(results).T
print(results_df)

                           MSE         MAE        R2
BaseLine         321158.711358  363.902197  0.164447
All Features     317294.529668  361.702840  0.174500
Model Selection  317294.348381  361.702422  0.174501
